In [0]:

from pyspark.sql import functions as F

# 1. Define the range of time you want to cover
start_date = "2026-05-01"
end_date = "2030-12-31"

# 2. Generate a sequence of dates
time_dim_df = (spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) as date_raw"))

# 3. Extract attributes
gold_time_dim = (time_dim_df
    .withColumn("date_key", F.date_format("date_raw", "yyyyMMdd").cast("int"))
    .withColumn("full_date", F.col("date_raw"))
    .withColumn("day_of_week", F.dayofweek("date_raw"))
    .withColumn("day_name", F.date_format("date_raw", "EEEE"))
    .withColumn("day_of_month", F.dayofmonth("date_raw"))
    .withColumn("month", F.month("date_raw"))
    .withColumn("month_name", F.date_format("date_raw", "MMMM"))
    .withColumn("quarter", F.quarter("date_raw"))
    .withColumn("year", F.year("date_raw"))
    .withColumn("is_weekend", F.when(F.col("day_of_week").isin(1, 7), True).otherwise(False))
    .withColumn("week_of_year", F.weekofyear("date_raw"))
)

# 4. Save to Gold Layer
gold_time_dim.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_gold.dim_date")

display(spark.table("workspace.amazon_fullfilment_gold.dim_date").limit(10))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_gold.fact_order_lifecycle (
    -- Business Keys
    order_id STRING NOT NULL,
    customer_id STRING NOT NULL,
    date_key INT NOT NULL,  -- Joins to dim_date

    -- Milestone Timestamps
    date_initial TIMESTAMP,
    date_picking TIMESTAMP,
    date_shipped TIMESTAMP,

    -- Calculated Metrics (Durations in minutes)
    minutes_to_pick INT,
    minutes_to_ship INT,
    total_fulfillment_time INT,

    -- Status Tracking
    current_status STRING,
    is_completed BOOLEAN, -- Helper for filtering out closed orders
    
    -- Metadata
    _last_updated_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY (date_key, order_id)
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true',
    'comment' = 'Cumulative snapshot fact table tracking order flow milestones.'
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.amazon_fullfilment_gold.dim_geography (
    geo_key INT NOT NULL,
    city STRING,
    province STRING,
    postal_code STRING, -- Optional: depending on how granular you want to be
    country STRING,     -- You can hardcode 'Canada' if that's your only market
    _last_updated TIMESTAMP
)
USING DELTA
CLUSTER BY (province, city);

In [0]:
from pyspark.sql import functions as F

# 1. Read the raw address data
raw_addresses = spark.table("workspace.amazon_fullfilment_silver.address_silver")

# 2. Extract unique locations
# We group by City and Province to create a unique list of geographies
geography_df = (raw_addresses
    .select("City", "Province", "Postal_Code")
    .distinct()
    # Create a unique Integer Key based on the location strings
    .withColumn("geo_key", F.abs(F.hash("City", "Province", "Postal_Code")).cast("int"))
    .withColumn("country", F.lit("Canada")) # Hardcoding for your context
    .withColumn("_last_updated", F.current_timestamp())
)

# 3. Overwrite or Merge into Gold
geography_df.write.format("delta").mode("overwrite").saveAsTable("workspace.amazon_fullfilment_gold.dim_geography")

In [0]:
%sql
select * from workspace.amazon_fullfilment_gold.dim_geography